# Session-Based Observability for Multi-Turn Conversations

Group every span from a multi-turn chatbot by session and user ID so conversations appear as a single, filterable unit in the FutureAGI Tracing dashboard.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/future-agi/cookbooks/blob/cookbook/quickstart-notebooks/quickstart/session-observability.ipynb)
[![View on GitHub](https://img.shields.io/badge/View_on_GitHub-181717?logo=github&logoColor=white)](https://github.com/future-agi/cookbooks/blob/cookbook/quickstart-notebooks/quickstart/session-observability.ipynb)


By the end of this notebook you will have a multi-turn chatbot that tags every LLM span with a user ID and session ID, attaches per-turn metadata, and sends all traces to FutureAGI Tracing where each conversation appears as a grouped, filterable session.

**Prerequisites:**
- FutureAGI account → [app.futureagi.com](https://app.futureagi.com)
- API keys: `FI_API_KEY` and `FI_SECRET_KEY` ([Get your API keys](https://docs.futureagi.com/admin-settings))
- Python 3.9+
- OpenAI API key

## Install

In [ ]:
%pip install fi-instrumentation-otel traceai-openai openai --quiet

In [ ]:
import os

os.environ["FI_API_KEY"] = "your-api-key"          # Replace with your key
os.environ["FI_SECRET_KEY"] = "your-secret-key"    # Replace with your key
os.environ["OPENAI_API_KEY"] = "your-openai-api-key"  # Replace with your key

## Step 1: Register the tracer and instrument OpenAI

`register()` creates a tracer provider connected to FutureAGI. `OpenAIInstrumentor` patches the OpenAI client so every `chat.completions.create` call is captured automatically.

In [ ]:
import os
from fi_instrumentation import register
from fi_instrumentation.fi_types import ProjectType
from traceai_openai import OpenAIInstrumentor
from openai import OpenAI

# Connect to FutureAGI tracing
trace_provider = register(
    project_type=ProjectType.OBSERVE,
    project_name="chatbot-session-demo",
)

# Patch OpenAI so every call is traced automatically
OpenAIInstrumentor().instrument(tracer_provider=trace_provider)

client = OpenAI()

## Step 2: Tag a single request with user and session context

Wrap any OpenAI call with `using_user()` and `using_session()` context managers. Every span created inside the block automatically inherits the `user.id` and `session.id` attributes.

In [ ]:
from fi_instrumentation import using_user, using_session

user_id = "user-7f3a2b"
session_id = "session-c91d4e"

with using_user(user_id), using_session(session_id):
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": "Hello, what can you help me with?"}],
    )
    print(response.choices[0].message.content)

Go to [app.futureagi.com](https://app.futureagi.com) → **Tracing** → the span appears with `user.id` and `session.id` visible in the attributes panel.

## Step 3: Simulate a multi-turn conversation

A conversation loop where each turn calls OpenAI inside the same `using_user` and `using_session` block. All turns share identical `user.id` and `session.id` values, so they appear grouped in the dashboard.

In [ ]:
from fi_instrumentation import using_user, using_session

def run_conversation(user_id: str, session_id: str) -> None:
    """Run a 3-turn conversation. All spans share the same user and session IDs."""

    turns = [
        "What is photosynthesis?",
        "How does it differ from cellular respiration?",
        "Give me a one-sentence summary of both processes.",
    ]

    conversation_history = []

    with using_user(user_id), using_session(session_id):
        for turn_number, user_message in enumerate(turns, start=1):
            print(f"\n--- Turn {turn_number} ---")
            print(f"User: {user_message}")

            conversation_history.append({"role": "user", "content": user_message})

            response = client.chat.completions.create(
                model="gpt-4o-mini",
                messages=conversation_history,
            )

            assistant_message = response.choices[0].message.content
            conversation_history.append({"role": "assistant", "content": assistant_message})

            print(f"Assistant: {assistant_message}")


run_conversation(user_id="user-7f3a2b", session_id="session-c91d4e")

In the Tracing dashboard, click the **Sessions** tab (second tab after "LLM Tracing") to see all three turns grouped as a conversation.

> **Note:** The Sessions tab displays an auto-generated UUID as the session identifier — not the string you passed to `using_session()`. Your string is stored as the session `name` and used for grouping.

## Step 4: Add per-turn metadata

Use `using_metadata()` to attach structured data to each individual turn — turn number, conversation stage, or any context that helps you analyze quality trends later.

In [ ]:
from fi_instrumentation import using_user, using_session, using_metadata

def run_conversation_with_metadata(user_id: str, session_id: str) -> None:
    """Same conversation loop with per-turn metadata attached to each span."""

    turns = [
        {"message": "What is photosynthesis?",                       "stage": "opening"},
        {"message": "How does it differ from cellular respiration?",  "stage": "deepening"},
        {"message": "Give me a one-sentence summary of both processes.", "stage": "closing"},
    ]

    conversation_history = []

    with using_user(user_id), using_session(session_id):
        for turn_number, turn in enumerate(turns, start=1):
            user_message = turn["message"]
            conversation_history.append({"role": "user", "content": user_message})

            turn_metadata = {
                "turn_number": turn_number,
                "conversation_stage": turn["stage"],
                "total_turns": len(turns),
            }

            with using_metadata(turn_metadata):
                response = client.chat.completions.create(
                    model="gpt-4o-mini",
                    messages=conversation_history,
                )

            assistant_message = response.choices[0].message.content
            conversation_history.append({"role": "assistant", "content": assistant_message})

            print(f"Turn {turn_number} [{turn['stage']}]: {assistant_message[:80]}...")


run_conversation_with_metadata(user_id="user-7f3a2b", session_id="session-c91d4e")

Each span now carries `metadata` with `turnNumber`, `conversationStage`, and `totalTurns` — visible in the span detail panel.

> **Tip:** You can combine `using_user()`, `using_session()`, `using_metadata()`, and `using_tags()` into a single `using_attributes()` call.

## Step 5: Complete script — view grouped sessions

The complete script puts everything together. Run it and then open the Tracing dashboard to inspect the full session.

In [ ]:
import os
from fi_instrumentation import register, using_user, using_session, using_metadata
from fi_instrumentation.fi_types import ProjectType
from traceai_openai import OpenAIInstrumentor
from openai import OpenAI

# Setup tracing
trace_provider = register(
    project_type=ProjectType.OBSERVE,
    project_name="chatbot-session-demo",
)
OpenAIInstrumentor().instrument(tracer_provider=trace_provider)
client = OpenAI()

# Conversation data
USER_ID    = "user-7f3a2b"
SESSION_ID = "session-c91d4e"

turns = [
    {"message": "What is photosynthesis?",                            "stage": "opening"},
    {"message": "How does it differ from cellular respiration?",       "stage": "deepening"},
    {"message": "Give me a one-sentence summary of both processes.",   "stage": "closing"},
]

# Multi-turn loop
conversation_history = []

with using_user(USER_ID), using_session(SESSION_ID):
    for turn_number, turn in enumerate(turns, start=1):
        user_message = turn["message"]
        conversation_history.append({"role": "user", "content": user_message})

        with using_metadata({
            "turn_number": turn_number,
            "conversation_stage": turn["stage"],
            "total_turns": len(turns),
        }):
            response = client.chat.completions.create(
                model="gpt-4o-mini",
                messages=conversation_history,
            )

        assistant_message = response.choices[0].message.content
        conversation_history.append({"role": "assistant", "content": assistant_message})

        print(f"Turn {turn_number} [{turn['stage']}]")
        print(f"  User:      {user_message}")
        print(f"  Assistant: {assistant_message[:100]}...")
        print()

print(f"Session complete. View at: app.futureagi.com → Tracing → Sessions tab")

trace_provider.force_flush()

Navigate to the **Sessions** tab:

1. Open [app.futureagi.com](https://app.futureagi.com) → **Tracing** → select your project
2. Click the **Sessions** tab (second tab after "LLM Tracing")
3. Your session appears as a row showing total traces (3), duration, first/last messages
4. Click the session row to open the **conversation view** — all turns displayed as Human/AI pairs

> **Tip:** Use a unique `session_id` per conversation and a stable `user_id` per user (e.g., their database UUID).

## What you built

- Registered a FutureAGI tracer provider and auto-instrumented OpenAI
- Tagged every LLM span with `using_user()` and `using_session()` so spans are linked to a specific user and conversation
- Built a 3-turn chatbot loop where all spans share the same `session.id`
- Attached per-turn metadata (`turn_number`, `conversation_stage`) using `using_metadata()`
- Viewed grouped sessions in the **Sessions** tab

### Next steps

- [Manual Tracing](https://docs.futureagi.com/cookbook/quickstart/manual-tracing) — add custom spans, nest them into pipelines
- [Inline Evals in Tracing](https://docs.futureagi.com/cookbook/quickstart/inline-evals-tracing) — score each turn for faithfulness or toxicity
- [Agent Compass](https://docs.futureagi.com/cookbook/quickstart/agent-compass-debug) — surface agent failure patterns from session traces
- [Running Your First Eval](https://docs.futureagi.com/cookbook/quickstart/first-eval) — all built-in eval metrics